In [ ]:
from libs.dataset_loader import MulTweEmoDataset
import sklearn.metrics as skm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import json
import math

In [ ]:
def threshold_predictions(data, threshold):
    y_pred = np.zeros(data.shape)
    for i, pred in enumerate(data):
        y_pred[i] = pred > threshold
    return y_pred

def get_metrics(labels, data, target_names):
    results = skm.classification_report(labels, data, output_dict=True, zero_division=0, target_names=target_names)
    results = pd.DataFrame(results)
    results.columns = map(str.capitalize, results.columns)
    results = results.T.drop(columns="support")
    results.columns = map(str.capitalize, results.columns)
    return results

def plot_metrics(labels, data, target_names):
    results = get_metrics(labels, data, target_names)
    ax = pd.DataFrame(results).plot(kind="bar", figsize=(10,4), yticks=[x / 10 for x in range(0,11)])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right');
    ax.set_axisbelow(True)
    ax.yaxis.grid(True)
    
def metrics_to_latex(labels, data, target_names):
    results = get_metrics(labels, data, target_names)
    return(results.to_latex(float_format="%.4f"))

def sigmoid(x):
  return 1 / (1 + math.exp(-x))

In [ ]:
model_type = "high_support"
datasets = ["train", "val", "test"]
predictions = {}

classes = list(range(9))
drop_low_support=False
if model_type == "high_support":
    classes = [0,1,2,4,5,6]
    drop_low_support=True

load_dir = "./multimodal_results"

val, _ = MulTweEmoDataset.load(csv_path="./dataset/val_MulTweEmo.csv", drop_something_else=True, drop_low_support=drop_low_support, test_split=None)
test, _ = MulTweEmoDataset.load(csv_path="./dataset/test_MulTweEmo.csv", drop_something_else=True, drop_low_support=drop_low_support, test_split=None)
train, _ =  MulTweEmoDataset.load(csv_path="./dataset/train_MulTweEmo.csv", drop_something_else=True, drop_low_support=drop_low_support, test_split=None)
for set in datasets:

    with open(f"{load_dir}/{model_type}/{set}_predictions.np", "rb") as f:
        predictions[set] = np.load(f)

In [ ]:
# val_predictions = np.array(predictions["val"])
# test_predictions = np.array(predictions["test"])
# train_predictions = np.array(predictions["train"])
# emotions = MulTweEmoDataset.get_labels(drop_low_support=drop_low_support)

In [ ]:
# if model_type == "bert":
#     test = test.drop_duplicates(subset=["id"])
#     val = val.drop_duplicates(subset=["id"])
#     train = train.drop_duplicates(subset=["id"])
#     test_predictions = test_predictions[test.index]
#     val_predictions = np.vectorize(sigmoid)(val_predictions)
#     test_predictions = np.vectorize(sigmoid)(test_predictions)
#     train_predictions = np.vectorize(sigmoid)(train_predictions)
# if model_type == "text_only":
#     test = test.drop_duplicates(subset=["id"])
#     val = val.drop_duplicates(subset=["id"])
#     train = train.drop_duplicates(subset=["id"])
# if model_type == "base_augment":
#     train_predictions = train_predictions[:train.shape[0]]

In [ ]:
# val_labels = np.array(val["labels"].to_list())
# test_labels = np.array(test["labels"].to_list())
# train_labels = np.array(train["labels"].to_list())

# All final models comparisons

In [ ]:
test, _ = MulTweEmoDataset.load(csv_path="./dataset/test_MulTweEmo.csv", drop_something_else=True, test_split=None)
test_text_only, _ = MulTweEmoDataset.load(csv_path="./dataset/test_MulTweEmo.csv", drop_something_else=True, test_split=None)
test_text_only = test_text_only.drop_duplicates(subset=["id"])
test_high_support, _ = MulTweEmoDataset.load(csv_path="./dataset/test_MulTweEmo.csv", drop_something_else=True, drop_low_support=True, test_split=None)

test_labels = np.array(test["labels"].to_list())
test_text_only_labels = np.array(test_text_only["labels"].to_list())
test_high_support_labels = np.array(test_high_support["labels"].to_list())
emotions = MulTweEmoDataset.get_labels()
emotions_high_support = MulTweEmoDataset.get_labels(drop_low_support=True)

model_types = ["bert",
    "base",
    "base_captions",
    "base_augment",
    "high_support",
    "text_only",
    "zero-shot"]

model_names = ["BERT", "CLIP", "Captions CLIP", "Silver CLIP"]

storage_name = f"sqlite:///threshold_training_study.db"


# best_treshold_trials = {
#     "bert":456,
#     "base":419,
#     "base_captions":499,
#     "base_augment":455
# }
best_treshold_trials = {
    "bert":497,
    "base":323,
    "base_captions":158,
    "base_augment":474,
    # "high_support":184,
    "text_only": 414
}
predictions = {}
default_predictions = {}

for model_type in model_types:
    if model_type == "zero-shot":
        llava_results_path = "./zero_shot_results/test/results_3.np"
        with open(llava_results_path, "rb") as f:
            predictions["zero-shot"] = np.load(f)
    else:
        with open(f"multimodal_results/{model_type}/test_predictions.np", "rb") as f:
            predictions[model_type] = np.load(f)
            
        if model_type == "bert":
            predictions[model_type] = predictions[model_type][test_text_only.index]
            predictions[model_type] = np.vectorize(sigmoid)(predictions[model_type])


        study = optuna.create_study(study_name=f"{model_type}", storage=storage_name, load_if_exists=True, directions=["maximize", "maximize"])
        if model_type == "high_support":
            thresholds = 0.5
        else:
            trial = study.trials[best_treshold_trials[model_type]]
            thresholds = [t for t in trial.params.values()]
        default_predictions[model_type] = threshold_predictions(predictions[model_type], 0.5)
        predictions[model_type] = threshold_predictions(predictions[model_type], thresholds)
        

In [ ]:
predictions["bert"].shape

In [ ]:
results = {}
for model in model_types[:4]:
    results[model] = {}
    model_results = get_metrics(test_labels if model != "bert" else test_text_only_labels, predictions[model], emotions)
    for e in model_results.index:
        results[model][e] = model_results["F1-score"][e]
    # for e in emotions:
    #     results[model][e.capitalize()] = model_results["F1-score"][e.capitalize()]
        
ax = pd.DataFrame(results).plot(kind="bar", figsize=(10,4), yticks=[x / 10 for x in range(0,11)])
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right');
ax.set_axisbelow(True)
ax.yaxis.grid(True)
ax.legend(labels=model_names[:], fontsize=14)
# box = ax.get_position()
# ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
# # Put a legend to the right of the current axis
# ax.legend(loc='lower left', bbox_to_anchor=(1, 0.7))
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
# plt.savefig(f"final_models/comparisons/emotion_f1_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
results = {}
for model in model_types[1:4]:
    results[model] = {}
    model_results = get_metrics(test_labels if model != "bert" else test_text_only_labels, predictions[model], emotions)
    for e in model_results.index:
        results[model][e] = model_results["Precision"][e]
        
ax = pd.DataFrame(results).plot(kind="bar", figsize=(7,4), yticks=[x / 10 for x in range(0,11)])
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right');
ax.set_axisbelow(True)
ax.yaxis.grid(True)
ax.legend(labels=model_names[1:])
# box = ax.get_position()
# ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
# # Put a legend to the right of the current axis
# ax.legend(loc='lower left', bbox_to_anchor=(1, 0.7))
plt.savefig(f"final_models/comparisons/emotion_precision_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
results = {}
for model in model_types[1:4]:
    results[model] = {}
    model_results = get_metrics(test_labels if model != "bert" else test_text_only_labels, predictions[model], emotions)
    for e in model_results.index:
        results[model][e] = model_results["Recall"][e]
        
ax = pd.DataFrame(results).plot(kind="bar", figsize=(7,4), yticks=[x / 10 for x in range(0,11)])
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right');
ax.set_axisbelow(True)
ax.yaxis.grid(True)
ax.legend(labels=model_names[1:])
# box = ax.get_position()
# ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
# # Put a legend to the right of the current axis
# ax.legend(loc='lower left', bbox_to_anchor=(1, 0.7))
plt.savefig(f"final_models/comparisons/emotion_recall_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
np.array(test_labels).sum(axis=1).mean()

In [ ]:
# metrics = ["Precision", "Recall", "F1-score", "Accuracy", "No label samples"]
results = {}
# for model in model_types:
avg = "Samples avg"
for model in model_types[:4]:
    if model != "zero-shot":
            results[model+"_def"] = {}
            model_results = get_metrics(test_labels if model != "bert" else test_text_only_labels, default_predictions[model], emotions)
            results[model+"_def"]["Precision"] = model_results["Precision"]["Samples avg"]
            results[model+"_def"]["Recall"] = model_results["Recall"]["Samples avg"]
            results[model+"_def"]["F1-score"] = model_results["F1-score"]["Samples avg"]
            results[model+"_def"]["Accuracy"] = skm.accuracy_score(test_labels if model != "bert" else test_text_only_labels, default_predictions[model])
            results[model+"_def"]["Hamming"] = skm.hamming_loss(test_labels if model != "bert" else test_text_only_labels, default_predictions[model])
            
            unique, count = np.unique(default_predictions[model].sum(axis=1), return_counts=True)
            results[model+"_def"]["No labels"] = 0 if unique[0]!=0 else count[0]
            results[model+"_def"]["Average labels"] = default_predictions[model].sum(axis=1).mean()

    results[model] = {}
    model_results = get_metrics(test_labels if model != "bert" else test_text_only_labels, predictions[model], emotions)
    results[model]["Precision"] = model_results["Precision"]["Samples avg"]
    results[model]["Recall"] = model_results["Recall"]["Samples avg"]
    results[model]["F1-score"] = model_results["F1-score"]["Samples avg"]
    results[model]["Accuracy"] = skm.accuracy_score(test_labels if model != "bert" else test_text_only_labels, predictions[model])
    results[model]["Hamming"] = skm.hamming_loss(test_labels if model != "bert" else test_text_only_labels, predictions[model])
    
    unique, count = np.unique(predictions[model].sum(axis=1), return_counts=True)
    results[model]["No labels"] = 0 if unique[0]!=0 else count[0]
    results[model]["Average labels"] = predictions[model].sum(axis=1).mean()

results = pd.DataFrame(results)
results.columns = [col.capitalize().replace("_", " ") for col in results.columns]
results = results.T
results["No labels"] = results["No labels"].astype(int)
print(results.to_latex(float_format="%.4f", column_format="l|rrr|r|r|r|r"))

In [ ]:
metrics = ["Precision", "Recall", "F1-score", "Accuracy", "No label samples"]
results = {}
# for model in model_types:
avgs = ["Micro", "Macro", "Weighted", "Samples"]

model_types = [
    # "bert",
    "base",
    "base_captions",
    "base_augment",
    # "zero-shot",
    # "text_only"
    ]

for model in model_types:
    # if model != "zero-shot":
    #         results[model+"_def"] = {}
    #         model_results = get_metrics(test_labels if model != "bert" else test_bert_labels, default_predictions[model], emotions)
    #         results[model+"_def"]["Precision"] = model_results["Precision"]["Samples avg"]
    #         results[model+"_def"]["Recall"] = model_results["Recall"]["Samples avg"]
    #         results[model+"_def"]["F1-score"] = model_results["F1-score"]["Samples avg"]

    #         results[model+"_def"]["Accuracy"] = skm.accuracy_score(test_labels if model != "bert" else test_bert_labels, default_predictions[model])
    #         results[model+"_def"]["Hamming"] = skm.hamming_loss(test_labels if model != "bert" else test_bert_labels, default_predictions[model])
            
    #         unique, count = np.unique(default_predictions[model].sum(axis=1), return_counts=True)
    #         results[model+"_def"]["No labels"] = 0 if unique[0]!=0 else count[0]
    #         results[model+"_def"]["Average labels"] = default_predictions[model].sum(axis=1).mean()

    results[model] = {}
    model_results = get_metrics(test_text_only_labels if (model == "bert" or model == "text_only") else test_labels, predictions[model], emotions)
    for avg in avgs:
        results[model][avg + " Precision"] = model_results["Precision"][avg+ " avg"]
        results[model][avg + " Recall"] = model_results["Recall"][avg+ " avg"]
        results[model][avg + " F1-score"] = model_results["F1-score"][avg+ " avg"]
    results[model]["Accuracy"] = skm.accuracy_score(test_text_only_labels if (model == "bert" or model == "text_only") else test_labels, predictions[model])
    results[model]["Hamming"] = skm.hamming_loss(test_text_only_labels if (model == "bert" or model == "text_only") else test_labels, predictions[model])
    
    unique, count = np.unique(predictions[model].sum(axis=1), return_counts=True)
    results[model]["No labels"] = 0 if unique[0]!=0 else count[0]
    results[model]["Average labels"] = predictions[model].sum(axis=1).mean()

results = pd.DataFrame(results)
results.columns = [col.capitalize().replace("_", " ") for col in results.columns]
results = results
# results["No labels"] = results["No labels"].astype(int)
results = results.style.highlight_max(axis=1, props="textbf:--rwrap;")
results = results.format(formatter=lambda x: '%.4f' % x)
print(results.to_latex(column_format="l|r|r|r|r|r"))

In [ ]:
test_supports = np.array(test_labels).sum(axis=0).astype(int)
test_supports = np.append(test_supports,[test_supports.sum()]*4)
base_results = get_metrics(test_labels, predictions["base"], emotions)
llava_results = get_metrics(test_labels, predictions["zero-shot"], emotions)
base_results.columns = base_results.columns.map(lambda x: x+" ")
joined_results = pd.concat([llava_results, base_results], axis=1, join="inner")
joined_results["Support"] = test_supports
print(joined_results.to_latex(float_format="%.4f", column_format="l|rrr|rrr|r"))

In [ ]:
test_labels.sum(axis=1).mean()

In [ ]:
test_indices = np.nonzero(test_labels[:,[0,1,2,4,5,6]].sum(axis=1))[0]
test_supports = np.array(test_high_support_labels).sum(axis=0).astype(int)
test_supports = np.append(test_supports,[test_supports.sum()]*4)
base_results = get_metrics(test_high_support_labels, predictions["base"][test_indices][:, [0,1,2,4,5,6]], emotions_high_support)
llava_results = get_metrics(test_high_support_labels, predictions["high_support"], emotions_high_support)
base_results.columns = base_results.columns.map(lambda x: x+" ")
joined_results = pd.concat([llava_results, base_results], axis=1, join="inner")
joined_results["Support"] = test_supports
print(joined_results.to_latex(float_format="%.4f", column_format="l|rrr|rrr|r"))